In [ ]:
!pip install "numpy<2.0.0" "pandas<2.2.0" pytorch-lightning "qiskit==0.45.3" "qiskit-aer==0.13.3" "qiskit-ibm-runtime==0.19.1" torchquantum --quiet

In [15]:
import kagglehub
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
import torch.nn as nn
import torch.optim as optim
import torchquantum.functional as tqf
import torchquantum as tq
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [3]:
# Download latest version
path = kagglehub.dataset_download("ronanazarias/heart-desease-dataset")

print("Path to dataset files:", path)

100%|██████████| 53.6k/53.6k [00:00<00:00, 1.27MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ronanazarias/heart-desease-dataset/versions/1


In [4]:
# CARICAMENTO DATASET NEI TENSORI
def carica_e_prepara_heart_data(file_path):
    """Carica un CSV del dataset Heart Disease, rimuove la colonna indice,

    codifica le variabili categoriche e restituisce i tensori X e y.
    """
    # 1. Carica il file CSV
    df = pd.read_csv(file_path)

    # 2. Rimuove la prima colonna se è un contatore/indice indesiderato (es. Unnamed: 0)
    # Il CSV ha 13 colonne, la prima è l'indice e l'ultima è 'HeartDisease'
    if df.shape[1] == 13:
        df = df.iloc[:, 1:]  # Tiene solo dalla 2ª colonna in poi

    # 3. Separazione del target 'HeartDisease' dalle feature
    X_df = df.drop(columns=["HeartDisease"])
    y_series = df["HeartDisease"]

    # 4. Converte le colonne categoriche (testo) in numeri tramite One-Hot Encoding
    # pd.get_dummies trasforma ad es. Sex ('M', 'F') in due colonne separate con 0 e 1
    X_encoded = pd.get_dummies(
        X_df,
        columns=[
            "Sex",
            "ChestPainType",
            "RestingECG",
            "ExerciseAngina",
            "ST_Slope",
        ],
        drop_first=True,  # Evita la ridondanza nelle variabili dummy
    )

    # 5. Conversione in Numpy (Float32 per PyTorch)
    X_numpy = X_encoded.values.astype("float32")
    y_numpy = y_series.values.astype("int64")

    # 6. Creazione dei Tensori PyTorch
    X_tensor = torch.tensor(X_numpy)
    y_tensor = torch.tensor(y_numpy)

    return X_tensor, y_tensor

In [5]:
complete_path = "/root/.cache/kagglehub/datasets/ronanazarias/heart-desease-dataset/versions/1/Heart-disease/"

X_data1, y_data1 = carica_e_prepara_heart_data(complete_path + "/heart_part1.csv")
X_data2, y_data2 = carica_e_prepara_heart_data(complete_path + "/heart_part2.csv")

X_data_total = torch.cat((X_data1, X_data2), dim=0)
y_data_total = torch.cat((y_data1, y_data2), dim=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_data_total,
    y_data_total,
    test_size=0.15,
    random_state=42,
    stratify=y_data_total,  # Mantiene la stessa proporzione di sani/malati sia nel Train che nel Test!
)

print("--- CONTROLLO DIMENSIONI ---")
print(
    f"📦 X_train shape: {X_train.shape}"
)  # Dovrebbe essere (N_pazienti, N_features)
print(f"🏷️ y_train shape: {y_train.shape}")
print(
    f"📊 Numero di feature generate dopo l'encoding: {X_train.shape[1]} colonne in train e {X_test.shape[1]} colonne in test"
)

--- CONTROLLO DIMENSIONI ---
📦 X_train shape: torch.Size([780, 15])
🏷️ y_train shape: torch.Size([780])
📊 Numero di feature generate dopo l'encoding: 15 colonne in train e 15 colonne in test


In [7]:
class DatasetManager(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
# MODELLO QML
class QMLModel(pl.LightningModule):
  def __init__(self, input_dim, hidden_dim=16, n_qubits=4, num_classes=2, lr=0.001, class_weights=None):
    super(QMLModel, self).__init__()
    self.input_dim = input
    self.save_hyperparameters() # Salva i parametri per la tracciabilità
    self.lr = lr
    self.n_qubits = n_qubits
    self.qubits_list = list(range(n_qubits))

    # A. PARTE CLASSICA
    self.base_layer = nn.Linear(input_dim, hidden_dim) # restituisce 16 combinazioni con le colonne
    self.dropout_layer = nn.Dropout(0.2) # evita l'overfitting
    self.reductor_layer = nn.Linear(hidden_dim, n_qubits) # converte le 16 combinazioni in 4 numeri (qubits)

    # B. PARTE QUANTISTICA (TorchQuantum)
    self.q_device = tq.QuantumDevice(n_wires=n_qubits)
    self.quantum_layer = tq.RandomLayer(wires=self.qubits_list, n_ops=8)
    self.measure_layer = tq.MeasureAll(tq.PauliZ)

    # C. CLASSIFICATORE FINALE (Fusion Classica 16 + Quantistica 4 = 20)
    self.classifier = nn.Linear(hidden_dim + n_qubits, num_classes)

    # D. CALCOLO E BILANCIAMENTO PESI
    # Loss con pesi opzionali
    if class_weights is not None:
        self.register_buffer('weights', class_weights)
        self.criterion = nn.CrossEntropyLoss(weight=self.weights)
    else:
        self.criterion = nn.CrossEntropyLoss()

  def forward(self, x):
    # 1. Passaggio Classico
    h = torch.relu(self.base_layer(x)) # esegue una relu per dare variabilità ai dati

    d = self.dropout_layer(h) # evita l'overfitting

    q_angles = torch.tanh(self.reductor_layer(d)) * np.pi # 16 -> 4 in angoli pi-greco

    # 2. Encoding Quantistico e Circuito
    self.q_device.reset_states(x.shape[0]) # reset qubit a 0
    for i in range(self.n_qubits):
        tqf.ry(self.q_device, wires=i, params=q_angles[:, i]) # ruota i qubit sull'asse y


    self.quantum_layer(self.q_device) # applica le porte quantistiche
    final_q_measure = self.measure_layer(self.q_device) # misurazione finale

    # 3. Concatenazione (16 + 4 = 20) e Classificazione (2)
    combined = torch.cat((h, final_q_measure), dim=1)
    return self.classifier(combined)

  def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.criterion(preds, y)
        acc = (preds.argmax(dim=-1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

  def validation_step(self, batch, batch_idx):
      x, y = batch
      preds = self(x)
      loss = self.criterion(preds, y)
      acc = (preds.argmax(dim=-1) == y).float().mean()
      self.log('val_loss', loss, prog_bar=True)
      self.log('val_acc', acc, prog_bar=True)

  def configure_optimizers(self):
      # Ottimizzatore AdamW + Scheduler ReduceLROnPlateau per stabilità quantistica
      optimizer = optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
      scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
      return {
          "optimizer": optimizer,
          "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}
      }

In [9]:
# NORMALIZZAZIONE
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # restituisce numpy
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.joblib')

['scaler.joblib']

In [11]:
# CREAZIONE DATALOADERS
X_train_dataset = DatasetManager(X_train_scaled, y_train)
X_test_dataset = DatasetManager(X_test_scaled, y_test)

train_loader = DataLoader(X_train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(X_test_dataset, batch_size=32, shuffle=False)

In [12]:
# 1. Calcola i pesi in base a y_train
# Supponendo che y_train sia un tensore PyTorch o array numpy
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train.numpy()
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

print(f"Pesi calcolati per le classi (Sani / Malati): {class_weights_tensor}")

Pesi calcolati per le classi (Sani / Malati): tensor([1.1207, 0.9028])


In [13]:
# TRAINING

model = QMLModel(input_dim=X_train_scaled.shape[1], class_weights=class_weights_tensor)

trainer = pl.Trainer(max_epochs=15, accelerator="auto", log_every_n_steps=10)

trainer.fit(model, train_loader, test_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_layer     │ Linear           │    256 │ train │     0 │
│ 1 │ dropout_layer  │ Dropout          │      0 │ train │     0 │
│ 2 │ reductor_layer │ Linear           │     68 │ train │     0 │
│ 3 │ q_device       │ QuantumDevice    │      0 │ train │     0 │
│ 4 │ quantum_layer  │ RandomLayer      │      6 │ train │     0 │
│ 5 │ measure_layer  │ MeasureAll       │      0 │ train │     0 │
│ 6 │ classifier     │ Linear           │     42 │ train │     0 │
│ 7 │ criterion      │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 372                                                                                              
Non-trainable params: 0                                                                                            
Total params: 372                                                                                                  
Total estimated model params size (MB): 0.001                                                                      
Modules in train mode: 17                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.


In [16]:
# Esempio: Paziente di 55 anni con colesterolo e pressione elevati
sample_patient = {
    "Age": 55,
    "RestingBP": 140,
    "Cholesterol": 289,
    "FastingBS": 1,
    "MaxHR": 122,
    "Oldpeak": 1.5,
    "Sex_M": 1,
    "ChestPainType_ATA": 0,
    "ChestPainType_NAP": 0,
    "ChestPainType_TA": 0,
    "RestingECG_Normal": 0,
    "RestingECG_ST": 1,
    "ExerciseAngina_Y": 1,
    "ST_Slope_Flat": 1,
    "ST_Slope_Up": 0,
}

# 1. Prepara e scala i dati del paziente (ritorna un tensore [1, 15])
scaler = joblib.load("scaler.joblib")

# 1. Prepara e scala il dato (garantendo il formato 2D [1, 15])
patient_scaled = scaler.transform([list(sample_patient.values())]) # La doppia quadra [[...]] fa il reshape 2D
patient_tensor = torch.tensor(patient_scaled, dtype=torch.float32)

# 2. Mette il modello in modalità inferenza e passa il tensore direttamente
model.eval()
with torch.no_grad():
    logits = model(patient_tensor)  # logits sono i numeri finali Es.: [1.02 , 3.08]
    probs = torch.softmax(logits, dim=1) # converte i logits in 2 numeri la cui somma fa 1

risk_percent = round(probs[0][1].item() * 100, 2) # converte i valori in percentuali
label = "Alto Rischio" if probs.argmax(dim=1) == 1 else "Basso Rischio"

print(f"Diagnosi: {label} ({risk_percent}% rischio)")

Diagnosi: Alto Rischio (96.9% rischio)
